# Phase 0 / NB3 — Single-Window SABRE Router

**Where we are:** NB1 = noise channels, NB2 = hardware specs (`HW`, `TECHS`, `SABRE_SEEDS`).
NB3 does exactly one thing: **route a single residency-stable window onto a fixed coupling
map by inserting SWAPs**, and report what was inserted plus where every qubit ended up. It is
the only place SWAPs are computed. No noise, no fidelity — it builds the physically-honest
circuit that NB4 schedules and NB5 scores.

**Public function**
`route(circuit, coupling_map, seeds=SABRE_SEEDS) -> RouteResult(routed_circuit, swap_count, final_layout)`

**The five locked rules**
1. **All-to-all passthrough.** `coupling_map is None` (NA/TI) -> return circuit unchanged,
   0 swaps, identity layout. No routing entered.
2. **SABRE routing-only, best-of-N.** `SabreSwap(cm, heuristic="decay", seed=s)` for each
   seed; keep the fewest-swaps result (ties -> lowest seed). Never strawman SC with a bad seed.
3. **Insert swaps only.** Bare `PassManager([SabreSwap])` -- no basis translation, no gate
   cancellation, no optimization. `swap` gates stay as `swap` (NOT decomposed to 3 CX): NB4
   applies the `f2q^3` / `3*t2q` channel per swap directly.
4. **Return the final permutation.** `final_layout[i]` = physical wire where virtual qubit i
   ends up. NB5 uses it to un-permute before comparing to the ideal -- the one bookkeeping
   step that, if dropped, silently produces low fidelities on routed circuits only.
5. **Correctness invariant.** Routed circuit == original **up to `final_layout`**; asserted
   here at the statevector level via `PermutationGate(final_layout)`.

**Scope boundary (NOT NB3):** windowing, persistent placement across windows, residency
changes, move channels, non-identity initial layout. Those are NB4. For Phase 1 this never
bites -- 1A is static (window = whole circuit) and 1B's SC module is a single edge (cap 2, no
swaps ever). The `initial_layout` arg is a Phase-2 stub.

In [1]:
import numpy as np
from dataclasses import dataclass
from qiskit import QuantumCircuit
from qiskit.transpiler import CouplingMap, PassManager
from qiskit.transpiler.passes import SabreSwap
from qiskit.circuit.library import PermutationGate
from qiskit.quantum_info import Statevector, state_fidelity

# In the notebook series these come from NB2; redefined here so NB3 runs standalone.
SABRE_SEEDS = list(range(10))
def _ring(n):
    return CouplingMap([[i,(i+1)%n] for i in range(n)] + [[(i+1)%n,i] for i in range(n)])
print("standalone deps ready; SABRE_SEEDS =", SABRE_SEEDS)

standalone deps ready; SABRE_SEEDS = [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]


## 1. The router

In [2]:
@dataclass
class RouteResult:
    routed_circuit: QuantumCircuit
    swap_count: int
    final_layout: list        # final_layout[i] = physical wire where virtual qubit i ends

def _final_perm(prop_set, n):
    """Normalize SabreSwap's final_layout to a plain list [virtual i -> physical wire]."""
    fl = prop_set.get("final_layout")
    if fl is None:
        return list(range(n))                       # no swaps -> identity
    v2p = {q._index: p for q, p in fl.get_virtual_bits().items()}
    return [v2p[i] for i in range(n)]

def route(circuit, coupling_map, seeds=SABRE_SEEDS, initial_layout=None):
    """Route one residency-stable window. See module docstring for the five rules."""
    if initial_layout is not None:
        raise NotImplementedError(
            "non-identity initial_layout is a Phase-2 / NB4 concern (carried placement across "
            "windows); NB3 routes identity-initial windows only.")
    if any(inst.operation.name in ("measure", "reset") for inst in circuit.data):
        raise ValueError("strip terminal measurements/resets before routing (NB4/NB5 rule)")

    n = circuit.num_qubits
    # Rule 1: all-to-all -> passthrough
    if coupling_map is None:
        return RouteResult(circuit, 0, list(range(n)))

    # Rules 2+3: bare SabreSwap, best-of-N by min swaps (ties -> lowest seed)
    best = None
    for s in seeds:
        pm = PassManager([SabreSwap(coupling_map, heuristic="decay", seed=s)])
        tqc = pm.run(circuit)
        nsw = tqc.count_ops().get("swap", 0)
        if best is None or nsw < best[0]:
            best = (nsw, tqc, _final_perm(pm.property_set, n))
    nsw, tqc, perm = best
    return RouteResult(tqc, nsw, perm)

def un_permute(state, final_layout):
    """Undo routing permutation so a routed output aligns with the ideal qubit order.
    Works for Statevector and DensityMatrix (NB5 uses DensityMatrix)."""
    return state.evolve(PermutationGate(final_layout))

print("route() and un_permute() defined")

route() and un_permute() defined


## 2. Correctness invariant (the un-permutation bug, caught here)

Two checks. First a controlled **3-cycle** (non-involution, so direction is unambiguous):
routing permutes the output, raw fidelity drops, `un_permute(state, final_layout)` restores it,
and the *inverse* pattern does not — proving we use the right direction. Then the same through
the real `route()` path on an asymmetric circuit.

In [3]:
# (a) controlled 3-cycle: apply swap(0,1) then swap(1,2) -> final_perm [2,0,1,3]
U = QuantumCircuit(4)
for i, ang in enumerate([0.3, 0.7, 1.1, 1.9]):
    U.rx(ang, i)
U.cx(0, 1); U.cx(2, 3)
routed = U.copy(); routed.swap(0, 1); routed.swap(1, 2)

state = [0, 1, 2, 3]
for a, b in [(0, 1), (1, 2)]:
    state[a], state[b] = state[b], state[a]
final_perm = [state.index(v) for v in range(4)]
inverse = [final_perm.index(i) for i in range(4)]
print(f"3-cycle final_perm={final_perm} inverse={inverse} (non-involution: {final_perm!=inverse})")

sv_o, sv_r = Statevector(U), Statevector(routed)
print(f"  raw fidelity                 = {state_fidelity(sv_o, sv_r):.4f}  (<1: permutation matters)")
print(f"  un_permute(final_perm)       = {state_fidelity(sv_o, un_permute(sv_r, final_perm)):.4f}  (==1: correct)")
print(f"  un_permute(inverse) [wrong]  = {state_fidelity(sv_o, un_permute(sv_r, inverse)):.4f}")

3-cycle final_perm=[2, 0, 1, 3] inverse=[1, 2, 0, 3] (non-involution: True)
  raw fidelity                 = 0.4000  (<1: permutation matters)
  un_permute(final_perm)       = 1.0000  (==1: correct)
  un_permute(inverse) [wrong]  = 0.4000


In [4]:
# (b) same invariant through route() on an asymmetric circuit that forces swaps on the ring
asym = QuantumCircuit(4)
for i, ang in enumerate([0.3, 0.7, 1.1, 1.9]):
    asym.rx(ang, i)
for a, b in [(0, 2), (1, 3), (0, 1), (2, 3)]:      # 0-2, 1-3 non-adjacent -> swaps
    asym.cx(a, b)

r = route(asym, _ring(4))
raw = state_fidelity(Statevector(asym), Statevector(r.routed_circuit))
fixed = state_fidelity(Statevector(asym), un_permute(Statevector(r.routed_circuit), r.final_layout))
print(f"route(): swaps={r.swap_count} final_layout={r.final_layout}")
print(f"  raw fidelity (no un-permute) = {raw:.4f}")
print(f"  after un_permute             = {fixed:.4f}  (== 1.0)")

route(): swaps=1 final_layout=[0, 1, 3, 2]
  raw fidelity (no un-permute) = 0.1417
  after un_permute             = 1.0000  (== 1.0)


## 3. Routing behaviour by technology

Dense block on the SC ring inserts swaps and preserves logical gate counts; the same block on
an all-to-all module (`coupling_map=None`) inserts none and is returned unchanged.

In [5]:
# dense 6-CX block
dense = QuantumCircuit(4)
for a in range(4):
    dense.h(a)
for a, b in [(0, 1), (0, 2), (0, 3), (1, 2), (1, 3), (2, 3)]:
    dense.cx(a, b)

sc = route(dense, _ring(4))                # SC ring (cap 4)
aa = route(dense, None)                     # NA/TI all-to-all

def nonswap(c): return {k: v for k, v in c.count_ops().items() if k != "swap"}
print(f"SC ring : swaps={sc.swap_count:>2}  ops={dict(sc.routed_circuit.count_ops())}")
print(f"all2all : swaps={aa.swap_count:>2}  ops={dict(aa.routed_circuit.count_ops())}  unchanged={aa.routed_circuit is dense}")
print(f"logical gates preserved on SC route? {nonswap(dense) == nonswap(sc.routed_circuit)}")

# 1B sanity: SC module at cap 2 is a single edge -> a 2q gate is always adjacent, never swaps
edge = QuantumCircuit(2); edge.h(0); edge.cx(0, 1)
print(f"1B SC edge (cap 2): swaps={route(edge, CouplingMap([[0,1],[1,0]])).swap_count} (expect 0)")

SC ring : swaps= 2  ops={'cx': 6, 'h': 4, 'swap': 2}
all2all : swaps= 0  ops={'cx': 6, 'h': 4}  unchanged=True
logical gates preserved on SC route? True
1B SC edge (cap 2): swaps=0 (expect 0)


## 4. Checkpoint — NB3 go/no-go

NB4 imports `route`, `RouteResult`, `un_permute`.

In [6]:
def _checkpoint():
    ring = _ring(4)
    # (a) all-to-all passthrough
    aa = route(dense, None)
    assert aa.swap_count == 0 and aa.routed_circuit is dense and aa.final_layout == [0,1,2,3]
    # (b) dense SC block forces swaps
    sc = route(dense, ring)
    assert sc.swap_count > 0
    # (c) logical (non-swap) gate counts preserved exactly
    assert nonswap(dense) == nonswap(sc.routed_circuit)
    # (d) routed == original under final permutation, and the test is non-vacuous (raw < 1)
    raw = state_fidelity(Statevector(asym), Statevector(r.routed_circuit))
    fixed = state_fidelity(Statevector(asym), un_permute(Statevector(r.routed_circuit), r.final_layout))
    assert raw < 0.99 and np.isclose(fixed, 1.0, atol=1e-9)
    # (e) un-permute direction is correct (3-cycle: final_perm works, inverse does not)
    assert np.isclose(state_fidelity(sv_o, un_permute(sv_r, final_perm)), 1.0, atol=1e-9)
    assert state_fidelity(sv_o, un_permute(sv_r, inverse)) < 0.99
    # (f) cap-2 SC edge never swaps
    assert route(edge, CouplingMap([[0,1],[1,0]])).swap_count == 0
    # (g) Phase-2 stub is guarded
    try:
        route(dense, ring, initial_layout=[0,1,2,3]); raise AssertionError("stub not guarded")
    except NotImplementedError:
        pass
    return True

print("NB3 CHECKPOINT PASSED" if _checkpoint() else "FAILED")

NB3 CHECKPOINT PASSED
